# Stage 2 V2: Anti-Catastrophic Forgetting

## Problème du Stage 2 V1
- Accuracy globale : 80.52% → 45% (-35.5%)
- CGM et Healthy : 0% (perte totale)
- Le modèle prédit CMD pour presque tout

## Solution basée sur la recherche scientifique

| Paramètre | V1 (échec) | V2 (corrigé) |
|-----------|------------|---------------|
| Learning Rate | 1e-5 | **2e-6** (10x plus bas) |
| Warmup ratio | 0.1 | **0.15** |
| LoRA rank | r=8 | **r=32** |
| LoRA alpha | α=8 | **α=32** |
| Replay buffer | 0% | **15%** (300 samples Stage 1) |

### Stratégies anti-forgetting:
1. **LR 10x plus bas** → "prevents 50%+ of forgetting"
2. **Replay buffer 15%** → maintient les capacités de classification
3. **Rééquilibrage** → ~8k samples/classe

---

In [ ]:
# ============================================================
# CELL 1: IMPORTS
# ============================================================

import os
import json
import re
import random
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from dataclasses import dataclass
from typing import Dict, List, Any, Optional
from collections import defaultdict, Counter
from datasets import Dataset
from tqdm import tqdm

# IMPORTANT: Import unsloth AVANT transformers
from unsloth import FastVisionModel

from transformers import (
    AutoProcessor,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# ============================================================
# CELL 2: CONFIGURATION V2 ANTI-FORGETTING
# ============================================================

# Chemins
BASE_DIR = Path("/home/hounfodjidagba/learning/cassava/new_implementation")
STAGE1_DATA_DIR = BASE_DIR / "data" / "stage1"
STAGE2_DATA_PATH = BASE_DIR / "data" / "stage2" / "stage2_vqa_dataset.json"
CHECKPOINT_DIR = BASE_DIR / "checkpoints" / "stage2_v2_anti_forgetting"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Modèle de base
MODEL_NAME = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit"
TARGET_IMAGE_SIZE = (512, 512)

# ============================================================
# CONFIGURATION V2 - BASÉE SUR LA RECHERCHE SCIENTIFIQUE
# ============================================================

# LoRA Configuration - AMÉLIORÉE pour multi-tâche
LORA_CONFIG = {
    "r": 32,                # Augmenté de 8 → 32 pour multi-tâche
    "lora_alpha": 32,       # alpha = r (conservatif)
    "lora_dropout": 0.05,   # Légère régularisation
    "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"],  # Plus de modules
}

# Training Configuration - ANTI-FORGETTING
TRAINING_CONFIG = {
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,      # Effective batch = 16
    "num_train_epochs": 2,
    "learning_rate": 2e-6,                 # 10x PLUS BAS que V1 (1e-5)
    "warmup_ratio": 0.15,                  # Plus de warmup (0.1 → 0.15)
    "weight_decay": 0.01,
    "max_grad_norm": 0.5,
    "logging_steps": 25,
    "save_steps": 250,
    "eval_steps": 125,
    "bf16": torch.cuda.is_bf16_supported(),
    "fp16": not torch.cuda.is_bf16_supported(),
}

# Configuration replay buffer
REPLAY_CONFIG = {
    "n_samples": 300,           # 300 samples Stage 1
    "mix_ratio": 0.15,          # 15% du dataset final
}

# Configuration rééquilibrage
BALANCE_CONFIG = {
    "target_per_class": 8000,   # Cible par classe
}

# Classes
CLASS_NAMES = ["CBB", "CBSD", "CGM", "CMD", "Healthy"]
CLASS_FULL_NAMES = {
    "CBB": "Bactériose du manioc (CBB)",
    "CBSD": "Striure brune du manioc (CBSD)",
    "CGM": "Acarien vert du manioc (CGM)",
    "CMD": "Mosaïque du manioc (CMD)",
    "Healthy": "Sain"
}

print(f"{'='*60}")
print(f"CONFIGURATION STAGE 2 V2 - ANTI-FORGETTING")
print(f"{'='*60}")
print(f"\n📊 Différences clés vs V1:")
print(f"  - LR: 1e-5 → {TRAINING_CONFIG['learning_rate']} (10x plus bas)")
print(f"  - Warmup: 0.1 → {TRAINING_CONFIG['warmup_ratio']}")
print(f"  - LoRA r: 8 → {LORA_CONFIG['r']}")
print(f"  - Replay buffer: {REPLAY_CONFIG['n_samples']} samples ({REPLAY_CONFIG['mix_ratio']*100}%)")
print(f"  - Target/classe: {BALANCE_CONFIG['target_per_class']}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# CELL 3: CHARGEMENT MODÈLE BASE (nouveau LoRA)
# ============================================================

print("\n" + "="*60)
print("LOADING BASE MODEL WITH NEW LORA CONFIG")
print("="*60)

# Charger le modèle de base (PAS le checkpoint Stage 1)
# On part du modèle pré-entraîné avec un NOUVEAU LoRA configuré pour multi-tâche
model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

# Ajouter LoRA avec nouvelle configuration
model = FastVisionModel.get_peft_model(
    model,
    r=LORA_CONFIG["r"],
    lora_alpha=LORA_CONFIG["lora_alpha"],
    lora_dropout=LORA_CONFIG["lora_dropout"],
    target_modules=LORA_CONFIG["target_modules"],
    use_rslora=False,
    use_gradient_checkpointing="unsloth"
)

# Charger le processor
processor = AutoProcessor.from_pretrained(MODEL_NAME)

# Statistiques
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"\n✅ Modèle chargé avec LoRA V2")
print(f"   - Rank: {LORA_CONFIG['r']} (vs 8 en V1)")
print(f"   - Modules: {LORA_CONFIG['target_modules']}")
print(f"   - Trainable: {trainable:,} ({100*trainable/total:.4f}%)")
print("="*60)

In [ ]:
# ============================================================
# CELL 4: CRÉATION REPLAY BUFFER (Stage 1 → VQA)
# ============================================================

print("\n" + "="*60)
print("CREATING REPLAY BUFFER FROM STAGE 1")
print("="*60)

def load_stage1_data():
    """Charge les données Stage 1."""
    train_path = STAGE1_DATA_DIR / "cassava_train_balanced.json"
    with open(train_path, 'r') as f:
        return json.load(f)

def extract_image_path(text):
    """Extrait le chemin de l'image."""
    match = re.search(r'<img>(.*?)</img>', text)
    return match.group(1) if match else None

def get_class_from_response(response: str) -> str:
    """Extrait la classe de la réponse Stage 1."""
    if "CBB" in response or "Bacterial Blight" in response:
        return "CBB"
    elif "CBSD" in response or "Brown Streak" in response:
        return "CBSD"
    elif "CGM" in response or "Green Mottle" in response:
        return "CGM"
    elif "CMD" in response or "Mosaic" in response:
        return "CMD"
    elif "Healthy" in response:
        return "Healthy"
    return "Unknown"

def convert_stage1_to_vqa(stage1_samples, n_samples=300):
    """
    Convertit les samples Stage 1 en format VQA pour le replay buffer.
    Question: "Quelle maladie affecte cette feuille de manioc ?"
    Réponse: "Cette feuille présente [classe]. [description courte]"
    """
    
    # Sélectionner n_samples équilibrés par classe
    samples_by_class = defaultdict(list)
    for s in stage1_samples:
        for turn in s['conversations']:
            if turn['from'] == 'assistant':
                cls = get_class_from_response(turn['value'])
                if cls != "Unknown":
                    samples_by_class[cls].append(s)
                break
    
    # Équilibrer: n_samples / 5 par classe
    per_class = n_samples // 5
    selected = []
    for cls in CLASS_NAMES:
        cls_samples = samples_by_class[cls]
        if len(cls_samples) >= per_class:
            selected.extend(random.sample(cls_samples, per_class))
        else:
            selected.extend(cls_samples)
    
    # Convertir en format VQA
    replay_samples = []
    
    # Templates de réponses VQA
    response_templates = {
        "CBB": "Cette feuille présente des symptômes de **Bactériose du manioc (CBB)**. On observe des taches angulaires brunes avec halo jaune.",
        "CBSD": "Cette feuille est atteinte de **Striure brune du manioc (CBSD)**. Les symptômes incluent des marbrures jaunes et des nécroses.",
        "CGM": "Cette feuille montre des dégâts d'**Acarien vert du manioc (CGM)**. On observe des taches chlorotiques et une déformation des feuilles.",
        "CMD": "Cette feuille présente les symptômes de la **Mosaïque du manioc (CMD)**. Les motifs mosaïque jaune-vert sont caractéristiques.",
        "Healthy": "Cette feuille de manioc est **saine**. Aucun symptôme de maladie n'est visible."
    }
    
    for s in selected:
        # Extraire image path et classe
        image_path = None
        cls = None
        for turn in s['conversations']:
            if turn['from'] == 'user':
                image_path = extract_image_path(turn['value'])
            elif turn['from'] == 'assistant':
                cls = get_class_from_response(turn['value'])
        
        if image_path and cls and cls != "Unknown":
            replay_sample = {
                "id": f"replay_{s['id']}",
                "conversations": [
                    {
                        "from": "user",
                        "value": f"Picture 1: <img>{image_path}</img>\nQuelle maladie affecte cette feuille de manioc ?"
                    },
                    {
                        "from": "assistant",
                        "value": response_templates[cls]
                    }
                ],
                "metadata": {
                    "source": "replay_stage1",
                    "class": cls,
                    "severity": "general",
                    "qa_type": "identification",
                    "generation_method": "replay"
                }
            }
            replay_samples.append(replay_sample)
    
    return replay_samples

# Charger Stage 1 et créer replay buffer
stage1_data = load_stage1_data()
print(f"Stage 1 data loaded: {len(stage1_data)} samples")

replay_buffer = convert_stage1_to_vqa(stage1_data, n_samples=REPLAY_CONFIG['n_samples'])
print(f"\n✅ Replay buffer créé: {len(replay_buffer)} samples")

# Vérifier distribution
replay_dist = Counter(s['metadata']['class'] for s in replay_buffer)
print("\n📊 Distribution replay buffer:")
for cls, count in replay_dist.most_common():
    print(f"  {cls}: {count}")

In [ ]:
# ============================================================
# CELL 5: CHARGEMENT ET RÉÉQUILIBRAGE DATASET STAGE 2
# ============================================================

print("\n" + "="*60)
print("LOADING AND BALANCING STAGE 2 DATASET")
print("="*60)

# Charger Stage 2
with open(STAGE2_DATA_PATH, 'r', encoding='utf-8') as f:
    stage2_data = json.load(f)

print(f"Stage 2 data loaded: {len(stage2_data)} samples")

# Analyser distribution originale
original_dist = Counter(s['metadata']['class'] for s in stage2_data)
print("\n📊 Distribution originale Stage 2:")
for cls, count in original_dist.most_common():
    print(f"  {cls}: {count} ({count/len(stage2_data)*100:.1f}%)")

# Rééquilibrer
def balance_dataset(data, target_per_class=8000):
    """Rééquilibre le dataset vers target_per_class samples par classe."""
    samples_by_class = defaultdict(list)
    for s in data:
        cls = s['metadata']['class']
        samples_by_class[cls].append(s)
    
    balanced = []
    for cls in CLASS_NAMES:
        cls_samples = samples_by_class[cls]
        n = len(cls_samples)
        
        if n >= target_per_class:
            # Sous-échantillonnage
            balanced.extend(random.sample(cls_samples, target_per_class))
            print(f"  {cls}: {n} → {target_per_class} (sous-échantillonnage)")
        else:
            # Oversampling
            multiplier = target_per_class // n + 1
            oversampled = (cls_samples * multiplier)[:target_per_class]
            balanced.extend(oversampled)
            print(f"  {cls}: {n} → {target_per_class} (oversampling {multiplier}x)")
    
    random.shuffle(balanced)
    return balanced

print("\n🔄 Rééquilibrage en cours...")
stage2_balanced = balance_dataset(stage2_data, target_per_class=BALANCE_CONFIG['target_per_class'])
print(f"\n✅ Stage 2 rééquilibré: {len(stage2_balanced)} samples")

# Vérifier nouvelle distribution
balanced_dist = Counter(s['metadata']['class'] for s in stage2_balanced)
print("\n📊 Distribution après rééquilibrage:")
for cls in CLASS_NAMES:
    count = balanced_dist[cls]
    print(f"  {cls}: {count} ({count/len(stage2_balanced)*100:.1f}%)")

In [ ]:
# ============================================================
# CELL 6: CRÉATION MIXED DATASET (Stage 2 + Replay)
# ============================================================

print("\n" + "="*60)
print("CREATING MIXED DATASET (STAGE 2 + REPLAY BUFFER)")
print("="*60)

# Calculer les proportions
total_samples = len(stage2_balanced) + len(replay_buffer)
replay_ratio = len(replay_buffer) / total_samples

print(f"\n📊 Composition du dataset mixte:")
print(f"  - Stage 2 (VQA): {len(stage2_balanced)} ({(1-replay_ratio)*100:.1f}%)")
print(f"  - Replay (Stage 1): {len(replay_buffer)} ({replay_ratio*100:.1f}%)")
print(f"  - Total: {total_samples}")

# Combiner
mixed_data = stage2_balanced + replay_buffer
random.shuffle(mixed_data)

# Analyser distribution finale
final_dist = Counter(s['metadata']['class'] for s in mixed_data)
source_dist = Counter(s['metadata']['source'] for s in mixed_data)

print("\n📊 Distribution par classe (final):")
for cls in CLASS_NAMES:
    count = final_dist[cls]
    print(f"  {cls}: {count} ({count/len(mixed_data)*100:.1f}%)")

print("\n📊 Distribution par source:")
for source, count in source_dist.most_common(5):
    print(f"  {source}: {count}")

print(f"\n✅ Mixed dataset créé: {len(mixed_data)} samples")

In [ ]:
# ============================================================
# CELL 7: PRÉPARATION DATASET ET DATA COLLATOR
# ============================================================

def format_sample_for_qwen(sample):
    """Formate un sample pour Qwen2.5-VL."""
    conversations = sample['conversations']
    
    # Extraire image path et textes
    image_path = None
    user_text = ""
    assistant_text = ""
    
    for turn in conversations:
        if turn['from'] == 'user':
            match = re.search(r'<img>(.*?)</img>', turn['value'])
            if match:
                image_path = match.group(1)
            user_text = re.sub(r'Picture \d+: <img>.*?</img>\n?', '', turn['value']).strip()
        elif turn['from'] == 'assistant':
            assistant_text = turn['value']
    
    return {
        "conversations": [
            {"role": "user", "content": f"Picture 1: <image>\n{user_text}"},
            {"role": "assistant", "content": assistant_text}
        ],
        "image_path": image_path,
        "id": sample['id'],
        "class_name": sample['metadata']['class'],
        "source": sample['metadata'].get('source', 'unknown')
    }

# Formater
print("\nFormatting data...")
all_formatted = [format_sample_for_qwen(s) for s in tqdm(mixed_data, desc="Formatting")]

# Filtrer images valides
valid_samples = [s for s in all_formatted if s['image_path'] and os.path.exists(s['image_path'])]
print(f"\n✅ Valid samples: {len(valid_samples)} / {len(all_formatted)}")

# Split train/val
random.seed(42)
random.shuffle(valid_samples)
split_idx = int(len(valid_samples) * 0.9)
train_formatted = valid_samples[:split_idx]
val_formatted = valid_samples[split_idx:]

print(f"Train: {len(train_formatted)}")
print(f"Val: {len(val_formatted)}")

# Créer datasets
train_dataset = Dataset.from_list(train_formatted)
eval_dataset = Dataset.from_list(val_formatted)

In [ ]:
# ============================================================
# CELL 8: DATA COLLATOR
# ============================================================

try:
    import albumentations as A
    HAS_ALBUMENTATIONS = True
    train_transforms = A.Compose([
        A.RandomResizedCrop(size=(512, 512), scale=(0.9, 1.0), p=0.5),
        A.HorizontalFlip(p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.2),
    ])
    print("✅ Albumentations - augmentation légère")
except ImportError:
    HAS_ALBUMENTATIONS = False
    train_transforms = None
    print("⚠️ Albumentations non disponible")

@dataclass
class QwenVQADataCollator:
    """Data collator pour VQA."""
    processor: Any
    transforms: Any = None
    is_training: bool = True
    max_length: int = 1536

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch_messages = []
        batch_images = []

        for feature in features:
            image_path = feature['image_path']
            image = None

            if image_path and os.path.exists(image_path):
                try:
                    image = Image.open(image_path).convert('RGB')
                    if self.is_training and self.transforms is not None:
                        image_np = np.array(image)
                        augmented = self.transforms(image=image_np)
                        image = Image.fromarray(augmented['image'])
                    image = image.resize(TARGET_IMAGE_SIZE, resample=Image.BILINEAR)
                except:
                    image = Image.new('RGB', TARGET_IMAGE_SIZE, color='white')
            else:
                image = Image.new('RGB', TARGET_IMAGE_SIZE, color='white')

            batch_images.append([image])

            conversations = feature['conversations']
            messages = []
            
            for turn in conversations:
                role = turn['role']
                content = turn['content']

                if role == 'user' and len(messages) == 0:
                    text_content = content.replace('<image>', '').replace('Picture 1:', '').strip()
                    messages.append({
                        "role": role,
                        "content": [
                            {"type": "image", "image": image},
                            {"type": "text", "text": text_content}
                        ]
                    })
                else:
                    messages.append({
                        "role": role,
                        "content": content.replace('<image>', '').strip()
                    })

            batch_messages.append(messages)

        batch_texts_full = []
        batch_texts_user = []

        for messages in batch_messages:
            full_text = self.processor.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            batch_texts_full.append(full_text)

            user_messages = [msg for msg in messages if msg["role"] == "user"]
            user_text = self.processor.tokenizer.apply_chat_template(
                user_messages, tokenize=False, add_generation_prompt=True
            )
            batch_texts_user.append(user_text)

        batch = self.processor(
            text=batch_texts_full,
            images=batch_images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=self.max_length
        )

        labels = batch["input_ids"].clone()

        for i in range(len(batch_texts_user)):
            user_encoding = self.processor(
                text=[batch_texts_user[i]],
                images=[batch_images[i]],
                return_tensors="pt",
                padding=False,
                truncation=True,
                max_length=self.max_length
            )
            user_length = user_encoding["input_ids"].shape[1]

            if user_length < labels[i].shape[0]:
                labels[i, :user_length] = -100
            else:
                labels[i, :-20] = -100

        batch["labels"] = labels
        return batch

train_collator = QwenVQADataCollator(processor=processor, transforms=train_transforms, is_training=True)
eval_collator = QwenVQADataCollator(processor=processor, transforms=None, is_training=False)

print("✅ Data collators créés")

In [ ]:
# ============================================================
# CELL 9: TRAINER V2
# ============================================================

class Stage2V2Trainer(Trainer):
    """Trainer pour Stage 2 V2."""
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
        return (loss, outputs) if return_outputs else loss

print("\n" + "="*60)
print("CONFIGURING TRAINER V2")
print("="*60)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    
    per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
    
    num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
    learning_rate=TRAINING_CONFIG["learning_rate"],
    warmup_ratio=TRAINING_CONFIG["warmup_ratio"],
    lr_scheduler_type="cosine",
    
    weight_decay=TRAINING_CONFIG["weight_decay"],
    max_grad_norm=TRAINING_CONFIG["max_grad_norm"],
    
    logging_steps=TRAINING_CONFIG["logging_steps"],
    save_steps=TRAINING_CONFIG["save_steps"],
    save_total_limit=3,
    
    eval_strategy="steps",
    eval_steps=TRAINING_CONFIG["eval_steps"],
    
    bf16=TRAINING_CONFIG["bf16"],
    fp16=TRAINING_CONFIG["fp16"],
    
    optim="adamw_8bit",
    
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    seed=42,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=8,
    early_stopping_threshold=0.001
)

trainer = Stage2V2Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=train_collator,
    processing_class=tokenizer,
    callbacks=[early_stopping],
)

effective_batch = TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']
total_steps = len(train_dataset) // effective_batch * TRAINING_CONFIG['num_train_epochs']

print(f"\n📊 Configuration V2:")
print(f"  - Train samples: {len(train_dataset):,}")
print(f"  - Effective batch: {effective_batch}")
print(f"  - Total steps: ~{total_steps:,}")
print(f"  - LR: {TRAINING_CONFIG['learning_rate']} (10x plus bas que V1!)")
print(f"  - Warmup: {TRAINING_CONFIG['warmup_ratio']} (plus élevé)")
print(f"  - LoRA rank: {LORA_CONFIG['r']} (vs 8 en V1)")
print("="*60)

In [ ]:
# ============================================================
# CELL 10: ENTRAÎNEMENT V2
# ============================================================

print("\n" + "="*60)
print("STARTING STAGE 2 V2 TRAINING - ANTI-FORGETTING")
print("="*60)
print(f"\n🎯 Stratégies anti-forgetting actives:")
print(f"  1. LR 10x plus bas: {TRAINING_CONFIG['learning_rate']}")
print(f"  2. Replay buffer: {len(replay_buffer)} samples ({REPLAY_CONFIG['mix_ratio']*100}%)")
print(f"  3. Dataset rééquilibré: ~{BALANCE_CONFIG['target_per_class']}/classe")
print(f"  4. LoRA étendu: r={LORA_CONFIG['r']}, modules={LORA_CONFIG['target_modules']}")
print(f"\n🎯 Objectifs:")
print(f"  - CMD: ≥80% (vs 95.1% V1)")
print(f"  - CBSD: ≥65% (vs 27.8% V1)")
print(f"  - CGM: ≥45% (vs 0% V1)")
print(f"  - CBB: ≥45% (vs 9.1% V1)")
print(f"  - Healthy: ≥55% (vs 0% V1)")
print(f"  - Global: ≥65% (vs 45% V1)")
print("="*60 + "\n")

train_result = trainer.train()

print("\n" + "="*60)
print("STAGE 2 V2 TRAINING COMPLETED")
print("="*60)
print(f"Final loss: {train_result.training_loss:.4f}")
print(f"Total steps: {train_result.global_step}")

In [ ]:
# ============================================================
# CELL 11: SAUVEGARDE
# ============================================================

print("\nSaving Stage 2 V2 model...")
final_model_path = CHECKPOINT_DIR / "final_model"
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

config = {
    "stage": "2_v2_anti_forgetting",
    "base_model": MODEL_NAME,
    "lora_config": LORA_CONFIG,
    "training_config": TRAINING_CONFIG,
    "replay_config": REPLAY_CONFIG,
    "balance_config": BALANCE_CONFIG,
    "training_loss": train_result.training_loss,
    "total_steps": train_result.global_step,
    "train_samples": len(train_dataset),
    "replay_samples": len(replay_buffer),
}

with open(CHECKPOINT_DIR / "training_config.json", 'w') as f:
    json.dump(config, f, indent=2)

print(f"✅ Stage 2 V2 model saved to: {final_model_path}")

In [ ]:
# ============================================================
# CELL 12: ÉVALUATION COMPARATIVE
# ============================================================

print("\n" + "="*60)
print("ÉVALUATION COMPARATIVE V1 vs V2")
print("="*60)

FastVisionModel.for_inference(model)

def generate_response(image_path: str, question: str, max_new_tokens: int = 256):
    """Génère une réponse."""
    try:
        image = Image.open(image_path).convert('RGB').resize(TARGET_IMAGE_SIZE)
    except:
        return "Erreur"
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question}
            ]
        }
    ]
    
    input_text = processor.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    inputs = processor(
        text=[input_text],
        images=[[image]],
        return_tensors="pt",
        padding=True,
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    
    response = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response.lower():
        response = response.lower().split("assistant")[-1].strip()
    return response

def extract_disease(response: str) -> str:
    """Extrait la maladie de la réponse."""
    response_lower = response.lower()
    if "cmd" in response_lower or "mosaïque" in response_lower:
        return "CMD"
    elif "cbsd" in response_lower or "striure" in response_lower:
        return "CBSD"
    elif "cbb" in response_lower or "bactériose" in response_lower:
        return "CBB"
    elif "cgm" in response_lower or "acarien" in response_lower:
        return "CGM"
    elif "sain" in response_lower or "healthy" in response_lower:
        return "Healthy"
    return "Unknown"

# Évaluer sur validation set
identification_question = "Quelle maladie affecte cette feuille de manioc ?"

print("\n📊 Évaluation sur 100 samples...")
eval_samples = random.sample(val_formatted, min(100, len(val_formatted)))

results = []
for sample in tqdm(eval_samples, desc="Evaluating"):
    response = generate_response(sample['image_path'], identification_question)
    predicted = extract_disease(response)
    actual = sample['class_name']
    results.append({
        'actual': actual,
        'predicted': predicted,
        'correct': predicted == actual
    })

correct = sum(1 for r in results if r['correct'])
total = len(results)

print(f"\n" + "="*60)
print(f"RÉSULTATS V2 vs V1")
print(f"="*60)
print(f"\n📊 Accuracy globale V2: {correct/total*100:.1f}% (V1: 45.0%)")

# Résultats V1 pour comparaison
v1_results = {"CMD": 95.1, "CBSD": 27.8, "CGM": 0.0, "CBB": 9.1, "Healthy": 0.0}

print("\n📈 Par classe (V2 vs V1):")
class_metrics = {}
for cls in CLASS_NAMES:
    cls_samples = [r for r in results if r['actual'] == cls]
    if cls_samples:
        cls_correct = sum(1 for r in cls_samples if r['correct'])
        acc = cls_correct / len(cls_samples) * 100
        class_metrics[cls] = acc
        v1_acc = v1_results.get(cls, 0)
        diff = acc - v1_acc
        status = "✅" if diff > 0 else "⚠️" if diff == 0 else "❌"
        print(f"  {status} {CLASS_FULL_NAMES[cls]}: {acc:.1f}% (V1: {v1_acc:.1f}%, Δ{diff:+.1f}%)")

unknown = sum(1 for r in results if r['predicted'] == 'Unknown')
print(f"\n🔍 Réponses non identifiées: {unknown}")

In [ ]:
# ============================================================
# CELL 13: SAUVEGARDER RÉSULTATS
# ============================================================

eval_results = {
    "stage": "2_v2_anti_forgetting",
    "total_samples": len(results),
    "global_accuracy_v2": correct / total * 100,
    "global_accuracy_v1": 45.0,
    "class_metrics_v2": class_metrics,
    "class_metrics_v1": v1_results,
    "improvement": (correct / total * 100) - 45.0,
}

with open(CHECKPOINT_DIR / "eval_results.json", 'w') as f:
    json.dump(eval_results, f, indent=2)

print(f"\n✅ Résultats sauvegardés")
print(f"\n" + "="*60)
print("CONCLUSION")
print("="*60)
improvement = (correct / total * 100) - 45.0
if improvement > 0:
    print(f"\n🎉 Amélioration de {improvement:+.1f}% vs V1!")
else:
    print(f"\n⚠️ Pas d'amélioration significative ({improvement:+.1f}%)")
print(f"\nCheckpoint: {CHECKPOINT_DIR / 'final_model'}")